# 🎹 Keystroke Acoustic Classifier - Google Colab Training Pipeline

This notebook guides you step-by-step through training the 27-class keystroke sound identification model (`A-Z` + `SPACE`) using public verified datasets (**Kaggle Keyboard Sound**, **SKAID**, and **Multi-Pressure**).

### Pipeline Overview:
1. **Environment Setup & GPU Check**
2. **Dataset Setup** (Mount Google Drive or Direct Upload)
3. **Dependency Installation** (`soundfile`, `librosa`, `av`, `lightgbm`, `xgboost`, etc.)
4. **Dataset Audit & Canonical Verification** (27 classes)
5. **Benchmark Suite Execution** (EXP-01 to EXP-20+)
6. **Production Model Training & Calibration** (ExtraTrees + Feature Set D + P2 Norm)
7. **Evaluation, Confusion Matrix & Metrics Visualization**
8. **Export & Download Production Artifacts**

## Step 1: Hardware Acceleration & Environment Setup
Check available GPU / CPU hardware on Google Colab.

In [ ]:
import torch
import os
import sys

print(f"PyTorch Version: {torch.__version__}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Active Device: {device}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")

## Step 2: Install Required Audio & ML Dependencies
Install PyAV (`av`) for M4A participant decoding, `soundfile`, `librosa`, `lightgbm`, `xgboost`, and `tabulate`. Also uninstalls HuggingFace `datasets` to avoid namespace conflicts.

In [ ]:
# Remove conflicting HuggingFace datasets library if present
!pip uninstall -y -q datasets
!pip install -q soundfile librosa av lightgbm xgboost scikit-learn pandas numpy matplotlib seaborn joblib tabulate

## Step 3: Mount Google Drive & Unzip Backend
Mount Google Drive and unzip the package.

In [ ]:
import os
import sys

from google.colab import drive
drive.mount('/content/drive')

!unzip -q "/content/drive/MyDrive/keystroke_colab_package.zip" -d /content/
%cd /content/keystroke_backend_v2

BASE_DIR = '/content/keystroke_backend_v2'
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)
sys.modules.pop('datasets', None)

# Ensure package __init__.py files exist
for folder in ['audio', 'datasets', 'evaluation', 'features', 'models', 'training']:
    init_path = os.path.join(BASE_DIR, folder, '__init__.py')
    with open(init_path, 'a') as f: pass

print(f"Current working directory: {os.getcwd()}")

## Step 4: Verify Datasets in `data/raw/`
Ensure the following 5 raw dataset zip files exist:
- `archive.zip` (Kaggle Keyboard Sound Dataset)
- `Participant Recordings_zip.zip` (SKAID M4A Audio)
- `Keystroke Logs_zip.zip` (SKAID Key Log CSVs)
- `dataset.zip` (Multi-Pressure Primary Dataset)
- `Keystrokes_Dataset.zip` (Multi-Pressure Secondary Dataset)

In [ ]:
data_raw = os.path.join(BASE_DIR, 'data', 'raw')
print(f"Data directory: {data_raw}")
if os.path.exists(data_raw):
    files = os.listdir(data_raw)
    print("Found files:")
    for f in files:
        sz_mb = os.path.getsize(os.path.join(data_raw, f)) / (1024 * 1024)
        print(f" - {f:35s} ({sz_mb:.2f} MB)")
else:
    print(f"ERROR: {data_raw} does not exist. Please check your upload path.")

## Step 5: Run Dataset Audit & Canonical Mapping
This runs the auditor to verify:
1. Strict 27 canonical classes (`A-Z` + `SPACE`)
2. Exclusion guard prevents any forbidden local HP recordings
3. Generates `DATASET_AUDIT.md`, `dataset_statistics.csv`, and `class_distribution.csv`

In [ ]:
from datasets.audit import DatasetAuditor
from datasets.canonical import verify_classes_json

classes_json_path = os.path.join(BASE_DIR, 'classes.json')
verify_classes_json(classes_json_path)
print("classes.json verification passed!")

auditor = DatasetAuditor(data_dir=data_raw, output_dir=BASE_DIR)
audit_res = auditor.run_audit()

print("\n--- AUDIT COMPLETE ---")
print(f"Total public events analyzed: {audit_res['all_events_count']}")

## Step 6: (Optional) Run Full Benchmark Suite (EXP-01 to EXP-20)
Compare ExtraTrees, SVM, Random Forest, HistGradientBoosting, LightGBM, and PyTorch CNN across Kaggle, SKAID, and Multi-Pressure datasets.

In [ ]:
# To run the complete benchmark suite, uncomment and run:
# from training.benchmark import ExperimentRunner
# runner = ExperimentRunner(data_dir=data_raw, output_dir=BASE_DIR)
# df_results = runner.run_all_required_experiments()
# print(df_results[['experiment_id', 'model', 'features', 'val_macro_f1', 'test_macro_f1', 'test_accuracy']])

## Step 7: Train Final Production Model
Trains the final classifier on all verified public datasets:
- **Model**: ExtraTrees (or HistGradientBoosting / SVM / PyTorch CNN)
- **Feature Set**: Set D (98-D multi-domain acoustic features: 13 MFCC + Deltas + Spectral Centroid/Rolloff/Flatness/Bandwidth/ZCR + Transient Rise Time + Peak/RMS)
- **Normalization**: P2 (Peak Normalization)
- **Calibration**: Sigmoid / Platt scaling on validation fold for reliable confidence scores
- **Exports**: `model.joblib`, `audio_config.json`, `feature_config.json`, `preprocessing_config.json`, `thresholds.json`, `model_metadata.json` into `production/`

In [ ]:
from training.train import train_production_model

prod_dir = os.path.join(BASE_DIR, 'production')

metadata = train_production_model(
    data_dir=data_raw,
    prod_dir=prod_dir,
    model_name="ExtraTrees",
    feature_set="D",
    normalization="P2",
    sample_rate=22050
)

print("\n=== PRODUCTION TRAINING COMPLETE ===")
import json
print(json.dumps(metadata, indent=2))

## Step 8: Visualizing Model Evaluation & Confusion Matrix
Plot the 27-class confusion matrix and classification metrics.

In [ ]:
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
from datasets.canonical import CANONICAL_CLASSES

# Load trained model
model_path = os.path.join(prod_dir, 'model.joblib')
model = joblib.load(model_path)
print(f"Loaded trained model from {model_path}")

# Display validation metrics from metadata
print("\n--- Validation Metrics ---")
for k, v in metadata.get("validation_metrics", {}).items():
    print(f"{k:20s}: {v:.4f}")

## Step 9: Download Trained Production Artifacts
Package the `production/` folder and download it directly to your local computer.

In [ ]:
import shutil
from google.colab import files

zip_output_path = '/content/keystroke_production_artifacts'
shutil.make_archive(zip_output_path, 'zip', prod_dir)

download_file = f"{zip_output_path}.zip"
print(f"Created archive: {download_file} ({os.path.getsize(download_file)/(1024*1024):.2f} MB)")

# Download to local browser
files.download(download_file)
print("Downloaded! Unzip this into your local 'keystroke_backend_v2/production/' directory.")